## Data Enrichment

Read the concatinated data

Handle Missing values

Create derived columns for further analysis

Save the data in data/JC

### Loading Libraries

In [8]:
import requests

import numpy as np
import pandas as pd
import plotly.express as px
import folium
import shutil
import glob
import os
from pathlib import Path
from urllib.request import urlretrieve
from urllib.error import HTTPError, URLError
from zipfile import ZipFile

In [9]:
citibike_df = pd.read_csv("../data/citibike/JC2025.csv")

### Step 7: Setting Dates

In [10]:
citibike_df['started_at'] = pd.to_datetime(citibike_df['started_at'],errors="coerce")
citibike_df['ended_at'] = pd.to_datetime(citibike_df['ended_at'],errors="coerce")

### Step 8: df overview

In [11]:
citibike_df.columns

Index(['ride_id', 'rideable_type', 'started_at', 'ended_at',
       'start_station_name', 'start_station_id', 'end_station_name',
       'end_station_id', 'start_lat', 'start_lng', 'end_lat', 'end_lng',
       'member_casual'],
      dtype='str')

### Step 9: Missing Values

In [12]:
missing_values = (
    citibike_df
    .isna()
    .sum()
    .reset_index()
)

missing_values.columns = ["column", "missing_count"]

missing_values["missing_share"] = (
    missing_values["missing_count"] / len(citibike_df)
)

missing_values.sort_values("missing_count", ascending=False)

,column,missing_count,missing_share
7,end_station_id,4265,0.004480
10,end_lat,3425,0.003597
11,end_lng,3425,0.003597
6,end_station_name,3128,0.003285
4,start_station_name,3,0.000003
5,start_station_id,3,0.000003
8,start_lat,2,0.000002
9,start_lng,2,0.000002
0,ride_id,0,0.000000
3,ended_at,0,0.000000


### Step 10: Ride Duration

In [16]:
citibike_df["ride_duration_minutes"] = (
    citibike_df["ended_at"] - citibike_df["started_at"]
).dt.total_seconds() / 60

In [17]:
citibike_df = citibike_df.dropna(
    subset=[
        "ride_id",
        "started_at",
        "ended_at",
        "start_lat",
        "start_lng",
        "end_lat",
        "end_lng"
    ]
)

citibike_df = citibike_df[
    (citibike_df["ride_duration_minutes"] > 1) &
    (citibike_df["ride_duration_minutes"] <= 24 * 60)
].copy()

### Step 11: Time Based Variables

In [18]:
citibike_df["date"] = citibike_df["started_at"].dt.date
citibike_df["month"] = citibike_df["started_at"].dt.to_period("M").astype(str)
citibike_df["month_name"] = citibike_df["started_at"].dt.month_name()
citibike_df["day_of_week"] = citibike_df["started_at"].dt.day_name()
citibike_df["hour"] = citibike_df["started_at"].dt.hour

In [15]:
def assign_season(month_number):
    if month_number in [12, 1, 2]:
        return "Winter"
    elif month_number in [3, 4, 5]:
        return "Spring"
    elif month_number in [6, 7, 8]:
        return "Summer"
    else:
        return "Autumn"


citibike_df["season"] = (
    citibike_df["started_at"]
    .dt.month
    .apply(assign_season)
)

In [19]:
citibike_df[
    [
        "started_at",
        "date",
        "month",
        "month_name",
        "day_of_week",
        "hour",
        "season"
    ]
].head()

,started_at,date,month,month_name,day_of_week,hour,season
0,2025-02-22 17:40:16.500,2025-02-22,2025-02,February,Saturday,17,Winter
1,2025-02-21 12:28:13.319,2025-02-21,2025-02,February,Friday,12,Winter
2,2025-02-01 14:17:43.272,2025-02-01,2025-02,February,Saturday,14,Winter
3,2025-02-22 11:36:29.292,2025-02-22,2025-02,February,Saturday,11,Winter
4,2025-02-28 22:56:26.546,2025-02-28,2025-02,February,Friday,22,Winter


In [20]:
citibike_df.info()

<class 'pandas.DataFrame'>
Index: 948635 entries, 0 to 952092
Data columns (total 20 columns):
 #   Column                 Non-Null Count   Dtype         
---  ------                 --------------   -----         
 0   ride_id                948635 non-null  str           
 1   rideable_type          948635 non-null  str           
 2   started_at             948635 non-null  datetime64[us]
 3   ended_at               948635 non-null  datetime64[us]
 4   start_station_name     948634 non-null  str           
 5   start_station_id       948634 non-null  str           
 6   end_station_name       947936 non-null  str           
 7   end_station_id         947804 non-null  str           
 8   start_lat              948635 non-null  float64       
 9   start_lng              948635 non-null  float64       
 10  end_lat                948635 non-null  float64       
 11  end_lng                948635 non-null  float64       
 12  member_casual          948635 non-null  str           
 13  

In [21]:
citibike_df.to_csv("../data/JC/JC2025_Enriched.csv", index = False)